# 001_prompt_evals.ipynb — Claude en Vertex AI

Generación de un dataset de evaluación en JSON, guardado en dataset.json.

La primera celda configura el cliente y las rutas. Las llamadas requieren un proyecto y credenciales Google reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Vertex AI: Application Default Credentials (ADC).
import os
from pathlib import Path

from anthropic import AnthropicVertex
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "GOOGLE" / "README.md").is_file()
)
google_root = repo_root / "GOOGLE"
load_dotenv(google_root / ".env", override=False)
exercise_dir = google_root / "PromptEvalsBuildingWithApi"
os.chdir(exercise_dir)

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise ValueError("Configura GOOGLE_CLOUD_PROJECT en GOOGLE/.env.")
client = AnthropicVertex(
    project_id=project_id,
    region=os.getenv("CLOUD_ML_REGION", "global"),
)
model = os.getenv("VERTEX_MODEL_ID", "claude-sonnet-4-6")


In [ ]:
# Conserva todos los bloques de las respuestas, incluidas herramientas y thinking.
from anthropic.types import Message


def message_content(message):
    content = message.content if isinstance(message, Message) else message
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
    elif not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita bloques de contenido.")
    else:
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text" and not block.get("text", "").strip():
                raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, message):
    messages.append({"role": "user", "content": message_content(message)})


def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message_content(message)})


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
    max_tokens=4000,
    json_mode=False,
):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if stop_sequences:
        params["stop_sequences"] = list(stop_sequences)
    if json_mode:
        system = (system or "") + "\nReturn only valid JSON. Do not include Markdown fences or commentary."
    if system:
        params["system"] = system
    if tools:
        params["tools"] = list(tools)
        if tool_choice:
            params["tool_choice"] = tool_choice
    message = client.messages.create(**params)
    return text_from_message(message)


In [ ]:
import json

def generate_dataset():
    prompt = """
    Generate 3 short evaluation tasks involving Google Cloud that can be solved
    with a single Python function, JSON object or regular expression.
    Return a JSON array of objects with task, format (python/json/regex)
    and solution_criteria. Return only valid JSON, without Markdown fences.
    """
    messages = []
    add_user_message(messages, prompt)
    return json.loads(chat(messages, json_mode=True))

dataset = generate_dataset()
with (exercise_dir / "dataset.json").open("w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
dataset
